Studi Kasus: Deteksi Dini Anomali Transaksi Finansial dan Pola Belanja Mencurigakan.

Latar Belakang & Masalah: Lembaga keuangan dan platform e-commerce memproses jutaan transaksi harian. Mayoritas transaksi berjalan wajar, namun sebagian kecil merupakan aktivitas penipuan (fraud), peretasan akun (account takeover), atau pencucian uang (money laundering). Pola kecurangan ini dinamis dan menyimpang secara multidimensi (misal: penarikan dana sangat besar pada dini hari dengan kecepatan transaksi/frekuensi yang melonjak mendadak).
Solusi: Menerapkan siklus data sense-making:
Data Wrangling & Feature Engineering: Menghitung logaritmik nilai transaksi, rasio aktivitas malam hari (night velocity), dan frekuensi transaksi harian.
AI Anomaly Detection: Menggunakan algoritma Isolation Forest untuk mengisolasi titik data anomali pada ruang berdimensi banyak tanpa memerlukan label awal (unsupervised).


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

def studi_kasus_1_data_sensemaking():
    np.random.seed(42)
    n_normal = 970
    n_anomali = 30

    # 1. Simulasi Data: Transaksi Normal vs Anomali
    # Pola Normal: Nominal moderat, jam kerja siang/sore, frekuensi rendah
    amt_normal = np.random.exponential(scale=150_000, size=n_normal) + 20_000
    hours_normal = np.random.choice(range(8, 22), size=n_normal)
    velocity_normal = np.random.poisson(lam=3, size=n_normal) + 1

    # Pola Anomali: Nominal masif, jam dini hari (01:00-04:00), lonjakan frekuensi
    amt_anomaly = np.random.uniform(7_000_000, 25_000_000, size=n_anomali)
    hours_anomaly = np.random.choice([1, 2, 3, 4], size=n_anomali)
    velocity_anomaly = np.random.poisson(lam=18, size=n_anomali)

    df = pd.DataFrame({
        'Transaction_Amount': np.concatenate([amt_normal, amt_anomaly]),
        'Hour_of_Day': np.concatenate([hours_normal, hours_anomaly]),
        'Daily_Velocity': np.concatenate([velocity_normal, velocity_anomaly]),
        'Is_Actual_Fraud': [0]*n_normal + [1]*n_anomali
    })

    # 2. Feature Engineering & Sense-Making
    # Transformasi logaritmik untuk mereduksi skewness nilai nominal
    df['Amount_Log'] = np.log1p(df['Transaction_Amount'])
    df['Night_Activity'] = df['Hour_of_Day'].apply(lambda x: 1 if x < 6 else 0)

    features = ['Amount_Log', 'Hour_of_Day', 'Daily_Velocity', 'Night_Activity']
    X = df[features]

    # 3. Pemodelan Deteksi Anomali dengan AI (Isolation Forest)
    model = IsolationForest(contamination=0.03, random_state=42)
    df['Anomaly_Flag'] = model.fit_predict(X)
    df['Is_Predicted_Anomaly'] = df['Anomaly_Flag'].apply(lambda x: 1 if x == -1 else 0)
    df['Anomaly_Score'] = -model.decision_function(X) # Nilai lebih tinggi = makin mencurigakan

    # 4. Ringkasan Evaluasi
    detected = df[(df['Is_Actual_Fraud'] == 1) & (df['Is_Predicted_Anomaly'] == 1)].shape[0]
    total_fraud = df['Is_Actual_Fraud'].sum()
    fp = df[(df['Is_Actual_Fraud'] == 0) & (df['Is_Predicted_Anomaly'] == 1)].shape[0]

    print("=== HASIL EVALUASI DETEKSI ANOMALI TRANSAKSI ===")
    print(f"Total Sampel: {len(df)} | Aktual Anomali: {total_fraud}")
    print(f"Fraud Terdeteksi (Recall): {detected}/{total_fraud} ({detected/total_fraud*100:.1f}%)")
    print(f"False Positives: {fp}")
    print("\nTop 5 Transaksi Paling Mencurigakan Berdasarkan Anomaly Score:")
    top_5 = df.sort_values(by='Anomaly_Score', ascending=False)[
        ['Transaction_Amount', 'Hour_of_Day', 'Daily_Velocity', 'Anomaly_Score']
    ].head(5)
    print(top_5.to_string(index=False))

if __name__ == '__main__':
    studi_kasus_1_data_sensemaking()

=== HASIL EVALUASI DETEKSI ANOMALI TRANSAKSI ===
Total Sampel: 1000 | Aktual Anomali: 30
Fraud Terdeteksi (Recall): 30/30 (100.0%)
False Positives: 0

Top 5 Transaksi Paling Mencurigakan Berdasarkan Anomaly Score:
 Transaction_Amount  Hour_of_Day  Daily_Velocity  Anomaly_Score
       1.659982e+07            4              28       0.135397
       1.004491e+07            3              30       0.130210
       1.103889e+07            4              23       0.125573
       2.129226e+07            2              21       0.113343
       2.157962e+07            4              10       0.110316
